# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and conforms to 
[MLCommons Croissant 1.0](https://mlcommons.org/croissant/).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata as Python object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"License: {getattr(metadata, 'license', None)}")


## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below, we inspect all record sets and their fields, always referencing by their `@id`.

In [ ]:
# List all record sets and their fields, referencing everything by `@id`
record_sets = list(dataset.record_sets())
print(f"Record Sets available ({len(record_sets)}):")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}, Name: {rs.name}")
    print("    Fields (by @id):")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, Name: {field.name}, dataType: {field.data_type}")
    print("")
if not record_sets:
    print("No explicit record sets listed. Using dataset default.")

# For this dataset, if record_sets is empty, mlcroissant exposes a default record set as main_records
default_rs_id = None
if not record_sets:
    # This fallback is for croissant's default, the data table is the dataset itself
    # Try dumping the records to see field structure
    try:
        record_sample = next(dataset.records())
        print("Sample record from default record set:")
        pprint.pprint(record_sample)
        default_rs_id = None # The user will use `record_set=None` below for data extraction
    except Exception as e:
        print('Could not obtain records:', e)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always reference entities by their `@id` as seen above.

If `record_sets` is empty above, use `record_set=None` when calling `dataset.records()` (the default/main data table). Otherwise, use one of the discovered record set `@id` values.

In [ ]:
# If there are no user-defined record sets, use the dataset default
record_set_ids = []
if record_sets:
    record_set_ids = [rs.id for rs in record_sets]
else:
    record_set_ids = [None]  # Using None loads the default/main data table

# Load all data to pandas DataFrames
dataframes = dict()
for rset_id in record_set_ids:
    # Each record is a dictionary of field @id -> value
    records = list(dataset.records(record_set=rset_id))
    if records:
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record_set=@id {rset_id}")
    else:
        print(f"No records found for record_set=@id {rset_id}")

# Peek at the fields/columns of the main dataframe
main_rs_id = record_set_ids[0]
print('Columns (@id):', dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We will process the main record set, filtering records, normalizing numeric fields, and grouping by a categorical field. All field/column references use their `@id`.

You can adjust the filter field, group field, and threshold below according to fields discovered above.

In [ ]:
# Choose fields by their @id for EDA. Update these according to your overview above.
df = dataframes[main_rs_id].copy()

# Example: find a numeric field (e.g., 'coverage', or 'cr:field/coverage_percentage')
# For this dataset, let's guess likely field ids (check the printed columns above!)
possible_numeric_fields = [c for c in df.columns if 'coverage' in c or 'count' in c or 'mw' in c.lower() or 'abundance' in c.lower() or df[c].dtype in [int, float]]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.select_dtypes(include=['float', 'int']).columns[0] if len(df.select_dtypes(include=['float', 'int']).columns) > 0 else df.columns[0]
print(f"Using numeric field for filtering and normalization: {numeric_field}")

# Set a threshold for filtering (e.g., > 10)
threshold = 10
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()

print(f"Filtered records where {numeric_field} > {threshold} (count: {filtered_df.shape[0]}):")
print(filtered_df.head(3))

# Normalization (Z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Find a suitable group field (categorical): typically containing 'type', 'group', or low unique counts
categorical_candidates = [c for c in df.columns if df[c].nunique() <= 10 and c != numeric_field]
if categorical_candidates:
    group_field = categorical_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)  # mean of numeric columns
    print(grouped_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    group_field = None
    print("No suitable group_field found for grouping.")


## 5. Visualization

Visualize the filtered and normalized distributions of the chosen numeric field from above using `matplotlib`, including histogram and (if group_field found) boxplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric_field
plt.figure(figsize=(8, 5))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, show boxplot
if group_field:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=filtered_df[group_field], y=pd.to_numeric(filtered_df[numeric_field], errors='coerce'))
    plt.title(f'{numeric_field} by {group_field}')
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion

In this notebook, you loaded and explored the FAIR^2 dataset of protein mass spectrometry results for extracellular vesicles from human mast cells using the `mlcroissant` library. You inspected the metadata, explored available fields/columns (referenced strictly by `@id`), filtered, normalized, grouped, and visualized data.

This approach enables reproducible and standards-based data FAIRness and supports deeper scientific analysis. Refer to field and record set `@id`s for robust, schema-compliant exploration and downstream automation.